# TF Serving によるモデルの配信（サービング）
## 学習目標
1. Keras モデルを SavedModel 形式でエクスポートする方法を学ぶ
2. SavedModel を読み込んで使用する方法を学ぶ
3. TensorFlow を使用してシグネチャをカスタマイズする方法を学ぶ
4. Agent Platform にモデルをデプロイする方法を学ぶ
5. デプロイされたモデルをオンライン推論およびバッチ推論で使用する方法を学ぶ

このラボでは、トレーニング後にモデルを配信（サービング）する方法を学びます。<br>

機械学習モデルの配信にはインフラストラクチャが必要です。Agent Platform は、セットアップとメンテナンスの手間を軽減する自動スケーリングサービスを提供することで、これをシンプルにします。

Agent Platform を使用するために、Keras モデルを SavedModel 形式でエクスポートし、Agent Platform にデプロイする方法を見ていきます。その過程で、シグネチャについて、それらをカスタマイズする方法、およびデプロイされたモデルから推論結果を取得する方法を学びます。

## セットアップ

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")



In [ ]:
import base64
import json
import shutil
from datetime import datetime

import keras
import keras_hub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import tensorflow as tf
from google.cloud import aiplatform
from oauth2client.client import GoogleCredentials

## モデルの構築とトレーニング
モデルのトレーニングはこのラボの焦点ではないため、シンプルな MobileNet ベースのモデルを作成し、転移学習を使用して迅速にトレーニングしましょう。

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = PROJECT + "-flowers"
FILE_DIR = f"gs://{BUCKET}/data"

CLASSES = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]

IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_CHANNELS = 3

BATCH_SIZE = 32

[create_tfrecords_at_scale.ipynb](https://github.com/GoogleCloudPlatform/asl-ml-immersion/blob/master/notebooks/image_models/solutions/create_tfrecords_at_scale.ipynb) ノートブックを実行していない場合は、以下のセルのコメントを解除して、`gs://asl-public-data` バケットからデータをコピーしてください。

In [ ]:
# !gcloud storage buckets create gs://{BUCKET}
# !gcloud storage cp gs://asl-public-data/data/flowers/tfrecords/* {FILE_DIR}

このデータセットには、TFRecords にシリアライズされた花の画像が含まれています。tf.data API を使用してデータを読み込み、解析します。

In [ ]:
!gcloud storage ls {FILE_DIR}

In [ ]:
TRAIN_PATTERN = FILE_DIR + "/train*"
EVAL_PATTERN = FILE_DIR + "/eval*"


def parse_example(example):
    feature_description = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.int64),
    }
    example = tf.io.parse_single_example(example, feature_description)
    example["image"] = tf.io.decode_jpeg(example["image"], channels=3)
    example["image"] = tf.image.resize(
        example["image"], [IMG_HEIGHT, IMG_WIDTH]
    )
    return example["image"], example["label"]


train_ds = (
    tf.data.TFRecordDataset(tf.io.gfile.glob(TRAIN_PATTERN))
    .map(parse_example)
    .batch(BATCH_SIZE)
)
eval_ds = (
    tf.data.TFRecordDataset(tf.io.gfile.glob(EVAL_PATTERN))
    .map(parse_example)
    .batch(10)
)

In [ ]:
backbone = keras_hub.models.Backbone.from_preset(
    "mobilenet_v3_large_100_imagenet_21k",
)

transfer_model = tf.keras.Sequential(
    [
        keras.Input(
            shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), name="mobilenet_input"
        ),
        keras.layers.Rescaling(scale=1.0 / 255),
        backbone,
        keras.layers.GlobalMaxPooling2D(),
        keras.layers.Dropout(rate=0.2),
        keras.layers.Dense(
            len(CLASSES),
            activation="softmax",
            kernel_regularizer=tf.keras.regularizers.l2(0.0001),
        ),
    ]
)

transfer_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
transfer_model.fit(
    train_ds,
    epochs=5,
    validation_data=eval_ds,
)

## SavedModel のエクスポートと読み込み
トレーニング済みモデルが用意できました。このセクションでは、モデルを SavedModel 形式でエクスポートします。

また、追加のサービング関数を使用してモデルを拡張する方法についても見ていきます。追加のサービング関数により、モデルの推論に前処理や後処理のロジックを提供できるようになります。

### SavedModel でのエクスポート
SavedModel には、トレーニング済みパラメータ（tf.Variable など）と計算グラフを含む完全なモデルプログラムが含まれます。SavedModel でエクスポートされたモデルを実行するのに元のモデル構築コードは必要ないため、LiteRT、TensorFlow.js、TensorFlow Serving での共有やデプロイに役立ちます。

以下の API を使用して、SavedModel 形式でモデルを保存および読み込むことができます。

- Keras Model API: Keras は、`format="tf_saved_model"` オプションを使用した [keras.Model.export](https://keras.io/api/models/model_saving_apis/export/) メソッドによる SavedModel 形式のエクスポートをサポートしています。また、"onnx" 形式もサポートしています。
- Keras ExportArchive: [ExportArchive](https://keras.io/api/models/model_saving_apis/export/#exportarchive-class) モジュールは、さまざまなサービングエンドポイントとそのシグネチャの設定など、より細かい制御をサポートしています。

まず、Keras API を使用してトレーニング済みモデルを SavedModel にエクスポートしましょう。

In [ ]:
shutil.rmtree("export", ignore_errors=True)
os.mkdir("export")
# 通常のモデルのシリアライズには、.keras 拡張子を付けて .save() を使用できます
transfer_model.save("export/flowers_model.keras")

transfer_model.export(
    "export/saved_model", format="tf_saved_model", verbose=False
)

ディレクトリを確認してみましょう。

In [ ]:
!ls export/saved_model

ディレクトリ内に複数のファイルがあることが確認できます。

- `saved_model.pb`: 実際の TensorFlow プログラムまたはモデルと、テンソル入力を受け取りテンソル出力を生成する関数を識別する一連の名前付きシグネチャを含む、SavedModel のメインファイルです。
- `keras_metadata.pb`: tf.keras.Model.save() 関数でのみ作成されるファイルです。Keras モデルに関するメタデータが含まれます。
- `variables` ディレクトリ: モデルのすべての変数が含まれます。
- `assets` ディレクトリ: SavedModel に必要なアセットと呼ばれる任意のファイルが含まれます。例えば、ルックアップテーブルの初期化に使用される語彙ファイルなどです。読み込み時、アセットおよびそれらに依存するシリアライズされた関数は、SavedModel ディレクトリ内の正しいファイルパスを参照します。

### `saved_model_cli` コマンドを使用した SavedModel の調査

ビルド済みの TensorFlow バイナリを通じて TensorFlow をインストールした場合、SavedModel CLI はパス名 `bin/saved_model_cli` でシステムにすでにインストールされています。

SavedModel CLI は、SavedModel に対する次の 2 つのコマンドをサポートしています。

- `show`: SavedModel から利用可能な計算を表示します。
- `run`: SavedModel から計算を実行します。

ここでは、デフォルトのタグ（`serve`）とシグネチャ（`serving_default`）を指定して、`saved_model_cli show` コマンドで SavedModel を調査してみましょう。

In [ ]:
!saved_model_cli show --dir export/saved_model --tag_set serve --signature_def serving_default

具体的な関数の説明が表示されました。

- 入力形状は、`'mobilenet_input'` という名前の 3 チャンネルを持つ 224x224 画像のバッチ（`-1`）です。
- 出力形状は、`'output_*'` という名前の（5 種類の花それぞれの確率を表す）5 つの浮動小数点値のバッチです。

### 読み込みと予測

SavedModel としてエクスポートされると、プログラム内でモデルを読み込んで使用できます。

Python プログラムの場合は、SavedModel の代わりに `keras.models.load_model` を呼び出して `.keras` ファイルを直接読み込むこともできます。

また、推論フェーズでは、（Web API の場合のように）トレーニングフェーズで使用した前処理済みの TFRecords データやバッチ化された `(224, 224, 3)` テンソルをモデルが常に受け取るとは限らない場合があります。<br>
例えば、モデルが画像データへのファイルパスを受け取るとします。その場合、モデルを呼び出す前に画像パスを処理する前処理操作を追加する必要があります。

In [ ]:
filenames = [
    "gs://asl-public-data/data/flowers/jpegs/10172567486_2748826a8b.jpg",
    "gs://asl-public-data/data/flowers/jpegs/10386503264_e05387e1f7_m.jpg",
    "gs://asl-public-data/data/flowers/jpegs/10391248763_1d16681106_n.jpg",
    "gs://asl-public-data/data/flowers/jpegs/10712722853_5632165b04.jpg",
    "gs://asl-public-data/data/flowers/jpegs/10778387133_9141024b10.jpg",
    "gs://asl-public-data/data/flowers/jpegs/112334842_3ecf7585dd.jpg",
]


def preprocess(img_bytes):
    img = tf.image.decode_jpeg(img_bytes, channels=IMG_CHANNELS)
    img = keras.ops.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    return img


def read_from_jpegfile(filename):
    img = tf.io.read_file(filename)
    img = preprocess(img)
    return img


serving_model = keras.models.load_model("export/flowers_model.keras")
input_images = [read_from_jpegfile(f) for f in filenames]

f, ax = plt.subplots(1, 6, figsize=(15, 15))
for idx, img in enumerate(input_images):
    ax[idx].imshow(img.numpy())
    batch_image = keras.ops.expand_dims(img, axis=0)
    batch_pred = serving_model.predict(batch_image)
    pred = batch_pred[0]
    pred_label_index = keras.ops.argmax(pred).numpy()
    pred_label = CLASSES[pred_label_index]
    prob = pred[pred_label_index]
    ax[idx].set_title(f"{pred_label} ({prob:.2f})")

### 追加のサービング関数の定義

では、毎回このようなボイラープレートの前処理コードを書く必要があるのでしょうか？<br>
答えは「いいえ」です！カスタムサービングシグネチャを追加することで、これらの追加の前処理関数または後処理関数を SavedModel 自体に組み込むことができます。

一度 SavedModel にコンパイルされると、同じグラフをエッジデバイス上、C++ コード内、JavaScript プログラム内など、さまざまな環境で実行できます。

このモデルを Web サーバーにデプロイし、Web API 経由で予測を提供すると仮定しましょう。

API クライアントにとって最も使いやすいシグネチャはどのようなものでしょうか？画像コンテンツのテンソルを送信してもらう代わりに、例えば単に GCS 上の JPEG ファイルパスを要求することができます。<br>
そして、5 つの確率のテンソルを返す代わりに、確率から抽出されたわかりやすい情報を返送できます。

In [ ]:
def postprocess(pred):
    top_prob = keras.ops.max(pred, axis=[1])
    pred_label_index = keras.ops.argmax(pred, axis=1)
    pred_label = keras.ops.take(
        keras.ops.convert_to_tensor(CLASSES), pred_label_index
    )

    # カスタム出力
    return {
        "probability": top_prob,
        "flower_type_int": pred_label_index,
        "flower_type_str": pred_label,
    }


def predict_from_filename(filenames):

    # カスタム前処理
    input_images = keras.ops.map(read_from_jpegfile, filenames)

    # モデル
    batch_pred = transfer_model(input_images)  # model.predict() と同じ

    # カスタム後処理
    processed = postprocess(batch_pred)
    return processed

さらに、Base64 でエンコードされた画像データを受信して前処理できる別のサービング関数を定義しましょう。<br>
これは、オンライン推論で生の画像データを送信したい場合に役立ちます。

In [ ]:
def predict_from_b64(img_bytes):

    # カスタム前処理
    input_images = keras.ops.map(preprocess, img_bytes)

    # モデル
    batch_pred = transfer_model(input_images)

    # カスタム後処理
    processed = postprocess(batch_pred)
    return processed

ここでは、`add_endpoint()` を使用して、これらのグラフを異なるキー名で SavedModel 形式にエクスポートしましょう。
- `serving_default`: SavedModel のデフォルトのキー名です。このキーを `predict_from_filename` サービング関数でオーバーライドします。
- 画像のバイト文字列を受け取る `predict_from_b64` を持つ新しいサービングシグネチャキー `predict_from_b64` を追加します。

In [ ]:
export_archive = keras.export.ExportArchive()

export_archive.track(transfer_model)

export_archive.add_endpoint(
    name="serving_default",
    fn=predict_from_filename,
    # この関数は 1 つの文字列値（ファイル名）を受け取ります。
    input_signature=[keras.InputSpec(shape=(None,), dtype="string")],
)

export_archive.add_endpoint(
    name="predict_base64",
    fn=predict_from_b64,
    # この関数は 1 つの文字列値（バイト文字列）を受け取ります。
    input_signature=[keras.InputSpec(shape=(None,), dtype="string")],
)

export_archive.write_out("export/flowers_model_with_signature", verbose=False)

新しい SavedModel のメタデータを確認してみましょう。

In [ ]:
!saved_model_cli show --dir export/flowers_model_with_signature --tag_set serve --signature_def serving_default

新しい `serving_default` がファイルパスを受け取り、3 つのキー（`flower_type_int`、`flower_type_str`、`probability`）を持つ辞書を返すようになったことがわかります。

この SavedModel を読み込んで使用してみましょう。<br>
Keras では、`TFSMLayer` レイヤーを使用してコンパイルされた SavedModel を読み込むことができます（TensorFlow では、`tf.saved_model.load` を使用して同じことができます）。

追加の前処理コードや後処理コードを記述する必要がなくなったことに注目してください。

In [ ]:
serving_fn = keras.layers.TFSMLayer(
    "export/flowers_model_with_signature", call_endpoint="serving_default"
)

pred = serving_fn(keras.ops.convert_to_tensor(filenames))

# カスタム出力の表示
for k in pred.keys():
    print(f"{k:15}: {pred[k].numpy()}")

これらの出力は、確率のベクトルよりも API クライアントにとって有用に見えます。

`predict_base64` シグネチャも確認してみましょう。

In [ ]:
!saved_model_cli show --dir export/flowers_model_with_signature --tag_set serve --signature_def predict_base64

## Agent Platform Inference

モデルをデプロイする準備が整いました！

このノートブックでは、スケーラブルな Agent Platform サービスにモデルをデプロイします。
Agent Platform はバッチ推論とオンライン推論の両方をサポートしています。

まず、SavedModel を Agent Platform にアップロードしましょう。

### Agent Platform Inference サービスへのモデルのアップロード

In [ ]:
TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")

REGION = "us-central1"
MODEL_DISPLAYNAME = f"flower_classifier-{TIMESTAMP}"

print(f"MODEL_DISPLAYNAME: {MODEL_DISPLAYNAME}")

# from https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/predictions/pre-built-containers
SERVING_CONTAINER_IMAGE_URI = (
    "us-docker.pkg.dev/vertex-ai-restricted/prediction/tf_opt-gpu.2-17:latest"
)

まず、SavedModel を GCS バケットにアップロードします。

In [ ]:
!gcloud storage cp -R export/flowers_model_with_signature gs://{BUCKET}/{MODEL_DISPLAYNAME}

Python SDK を使用してモデルをアップロードできます。

ここでは、`display_name`、SavedModel のパスである `artifact_uri`、およびモデルが実行されるコンテナ環境である `serving_container_image_uri` を指定しています（この場合は事前ビルド済みコンテナが選択されていますが、必要に応じてカスタムコンテナを使用することもできます）。

詳細については、[SDK ドキュメント](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_upload)を参照してください。

In [ ]:
uploaded_model = aiplatform.Model.upload(
    display_name=MODEL_DISPLAYNAME,
    artifact_uri=f"gs://{BUCKET}/{MODEL_DISPLAYNAME}",
    serving_container_image_uri=SERVING_CONTAINER_IMAGE_URI,
)

アップロード後、コンソール UI で「Agent Platform」->「モデル」をクリックしてモデルを確認できます。

### バッチ推論

バッチ推論では、大規模なデータセットをモデルに渡して、バッチとしてまとめて予測を実行できます。<br>

#### 推論ファイルの作成
[バッチ推論](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/predictions)サービスは、JSON Lines、TFRecords、CSV、またはシンプルなテキストファイルリスト形式を受け入れます。

ここでは、多数の画像ファイルパスを含むシンプルなデータセットを [JSON Lines](https://jsonlines.org/) 形式で作成します。

In [ ]:
files = !gcloud storage ls -r gs://asl-public-data/data/flowers/jpegs/*.jpg
print(len(files))

In [ ]:
JSON_FILE = "batch_prediction.jsonl"

with open(JSON_FILE, "w") as f:
    for file in files:
        f.write(json.dumps({"filenames": file}) + "\n")

!head {JSON_FILE}

In [ ]:
!gcloud storage cp {JSON_FILE} {FILE_DIR}

#### バッチ推論ジョブの送信
[`aiplatform.batch_predict()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_batch_predict) 関数を使用してバッチ推論ジョブを呼び出しましょう。

必要に応じてマシンタイプとアクセラレータを指定できることに注意してください。<br>
これは、限られた時間内に大量のデータを処理したい場合に非常に便利です。

In [ ]:
TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")

JOB_DISPLAY_NAME = "flower_classification_batch"
MACHINE_TYPE = "n1-standard-4"

batch_pred_job = uploaded_model.batch_predict(
    job_display_name=JOB_DISPLAY_NAME,
    gcs_source=f"{FILE_DIR}/{JSON_FILE}",
    gcs_destination_prefix=f"{FILE_DIR}/batch_prediction_result/{TIMESTAMP}",
    machine_type=MACHINE_TYPE,
    accelerator_type="NVIDIA_TESLA_T4",
    accelerator_count=1,
    sync=False,
)

**約 20 分かかることに注意してください。完了するまで待つか、「オンライン推論」セクションに進んで後で次のセルに戻ってください。ステータスは「Agent Platform」->「モデル」->「バッチ推論」ページで確認できます。**

In [ ]:
if batch_pred_job.output_info:
    output_dir = batch_pred_job.output_info.gcs_output_directory
    results = !gcloud storage cat {output_dir}/prediction.results*
    for r in results[:5]:
        r = json.loads(r)
        print(f"filename       : {r['instance']['filenames']}")
        for k in r["prediction"].keys():
            print(f"{k:15}: {r['prediction'][k]}")
        print("*" * 30)
else:
    print(f"This job is still running.")

### オンライン推論

オンライン推論オプションでは、モデル専用のエンドポイントを作成し、それを Web API として使用できます。

[`aiplatform.Model.deploy`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_deploy) 関数を使用してエンドポイントを作成し、モデルをエンドポイントにリンクしましょう。ここでも、マシンタイプとアクセラレータを指定できます。

**以下のコマンドの実行には約 10 分かかります。**

In [ ]:
endpoint = uploaded_model.deploy(
    machine_type=MACHINE_TYPE,
    accelerator_type="NVIDIA_TESLA_T4",
    accelerator_count=1,
)

デプロイ後、エンドポイントを呼び出して結果を取得するだけです。<br>
エンドポイントの詳細は、「Agent Platform」->「エンドポイント」ページにアクセスして確認できます。

ここでは [Python SDK](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Endpoint#google_cloud_aiplatform_Endpoint_predict) を使用しますが、エンドポイントは任意の環境から呼び出すことができることに注意してください。

In [ ]:
instances = [{"filenames": f} for f in filenames]

In [ ]:
pred = endpoint.predict(instances=instances)

# カスタム出力の表示
for p in pred.predictions:
    for k in p.keys():
        print(f"{k:15}: {p[k]}")
    print("*" * 30)

### 生の画像を使用したオンライン予測

次に、`predict_base64` シグネチャを呼び出して生の画像データを渡してみましょう。<br>
API 経由で生の画像データを送信するには、データを Base64 でエンコードし、次のように b64 をキーとする JSON オブジェクトにカプセル化する必要があります。

```python
{ "b64": <base64 エンコードされた文字列> }
```

In [ ]:
# GCS からサンプル画像をダウンロード。
!gcloud storage cp gs://asl-public-data/data/flowers/jpegs/10172567486_2748826a8b.jpg sample.jpg


def b64encode(filename):
    with open(filename, "rb") as ifp:
        img_bytes = ifp.read()
        return base64.b64encode(img_bytes).decode()


data = {
    "signature_name": "predict_base64",
    "instances": [{"img_bytes": {"b64": b64encode("./sample.jpg")}}],
}

Python SDK の `endpoint.predict` は `serving_default` シグネチャのみをサポートしているため、ここでは一般的な Python の `requests` モジュールを使用して API 呼び出しを直接定義し、代わりに `rawPredict` API を使用して他のシグネチャを指定しましょう。

そのためには、認証トークンを定義し、それをリクエストヘッダーに含める必要があります。

In [ ]:
token = (
    GoogleCredentials.get_application_default().get_access_token().access_token
)
headers = {"Authorization": "Bearer " + token}

エンドポイントの URL は `https://<region>-aiplatform.googleapis.com/v1/projects/<project id>/locations/<region>/endpoints/<endpoint id>:rawPredict` です。<br>
それに応じて定義し、エンコードされた生の画像を API に送信しましょう。

In [ ]:
api = "https://{}-aiplatform.googleapis.com/v1/projects/{}/locations/{}/endpoints/{}:rawPredict".format(
    REGION, PROJECT, REGION, endpoint.name
)

response = requests.post(api, json=data, headers=headers)
json.loads(response.content)

これで、Agent Platform のオンライン予測サービスから結果を取得できました。

## まとめ
ここでは以下の方法を学びました。
- SavedModel のエクスポートと読み込み
- SavedModel の動作を制御するためのサービング関数のカスタマイズ
- SavedModel の Agent Platform へのデプロイ
- デプロイされたモデルのバッチ推論およびオンライン推論での使用

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.